## Preprocessing

In [1]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import harmonypy as hm
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# Optional Harmony
HAVE_HARMONY = True

# Plot defaults
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=1200, facecolor='white', fontsize=8)
mpl.rcParams['savefig.dpi'] = 1200
mpl.rcParams['savefig.bbox'] = 'tight'


In [2]:
# --------------------
# Parameters
# --------------------
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"
PBMC_H5AD = os.path.join(ROOT, "data", "SCEs", "analysis", "pbmc.h5ad")
BAL_H5AD  = os.path.join(ROOT, "data", "SCEs", "analysis", "bal.h5ad")

CONDITION_KEY = "Condition2"
BATCH_KEY = "sampleID"
CLUSTER_KEY = "ann_lv2"

COUNTS_LAYER = "counts"

N_HVG = 2000
N_PCS = 20
N_NEIGHBORS = 15

OUTDIR = "paga_pbmc_bal_by_condition2"
os.makedirs(OUTDIR, exist_ok=True)
sc.settings.figdir = OUTDIR


## Helpers: integer counts layer

In [3]:
def fraction_non_integer_like(X, tol=1e-6, sample_n=200000, seed=0):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return 0.0
    rng = np.random.default_rng(seed)
    if vals.size > sample_n:
        vals = vals[rng.choice(vals.size, size=sample_n, replace=False)]
    return float(np.mean(np.abs(vals - np.round(vals)) > tol))


def ensure_counts_layer_from_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER, tol=1e-6, max_frac_nonint=1e-4):
    if layer_key in adata.layers:
        return
    frac = fraction_non_integer_like(adata.X, tol=tol)
    print(f"fraction non-integer-like in adata.X: {frac:.6g}")
    if frac > max_frac_nonint:
        raise ValueError(
            f"adata.X is not integer-like enough to treat as raw counts (frac_nonint={frac}).\n"
            "Re-convert/export with counts preserved is recommended."
        )
    if sp.issparse(adata.X):
        Xc = adata.X.copy()
        Xc.data = np.rint(Xc.data).astype(np.int32)
        adata.layers[layer_key] = Xc
    else:
        adata.layers[layer_key] = np.rint(np.asarray(adata.X)).astype(np.int32)


def set_counts_as_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER):
    if layer_key not in adata.layers:
        raise KeyError(f"Missing layer '{layer_key}'")
    adata.X = adata.layers[layer_key].copy()
    return adata


## Load PBMC + BAL

In [4]:
from pathlib import Path
out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis")

out_file = out_dir / "mdm_processed.h5ad"
# save (AnnData.write is the common method; write_h5ad is also available in some versions)
adata = sc.read(str(out_file))

In [5]:
adata_pbmc = sc.read_h5ad(PBMC_H5AD)
adata_bal = sc.read_h5ad(BAL_H5AD)

for name, ad in [("pbmc", adata_pbmc), ("bal", adata_bal)]:
    for k in [CONDITION_KEY, BATCH_KEY, CLUSTER_KEY]:
        if k not in ad.obs.columns:
            raise KeyError(f"[{name}] Expected obs column '{k}' not found.")
    print(f"{name}: shape={ad.shape} layers={list(ad.layers.keys())}")
    print(f"{name}: X min/max={float(ad.X.min())} {float(ad.X.max())}")

print("Ensuring counts layers...")
ensure_counts_layer_from_X(adata_pbmc, layer_key=COUNTS_LAYER)
ensure_counts_layer_from_X(adata_bal, layer_key=COUNTS_LAYER)
print("pbmc layers:", list(adata_pbmc.layers.keys()))
print("bal layers:", list(adata_bal.layers.keys()))


pbmc: shape=(110809, 31166) layers=[]
pbmc: X min/max=0.0 17935.0
bal: shape=(107598, 31717) layers=[]
bal: X min/max=0.0 6411.0
Ensuring counts layers...
fraction non-integer-like in adata.X: 0
fraction non-integer-like in adata.X: 0
pbmc layers: ['counts']
bal layers: ['counts']


In [6]:
# -------------------------
# Normalize and Log-Transform
# -------------------------
for dataset in [adata_pbmc, adata_bal]:
    sc.pp.normalize_total(dataset, target_sum=1e4)
    sc.pp.log1p(dataset)


normalizing counts per cell
    finished (0:00:01)
normalizing counts per cell
    finished (0:00:01)


In [10]:
# # write to Hieu
# # create directory (including parents) if needed, then save AnnData
# from pathlib import Path

# out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis")
# out_dir.mkdir(parents=True, exist_ok=True)   # create directory if it doesn't exist

# out_file = out_dir / "bal_shared.h5ad"
# # save (AnnData.write is the common method; write_h5ad is also available in some versions)
# adata_bal.write(str(out_file))
# print(f"Saved AnnData to {out_file}")

# out_file = out_dir / "pbmc_shared.h5ad"
# # save (AnnData.write is the common method; write_h5ad is also available in some versions)
# adata_pbmc.write(str(out_file))
# print(f"Saved AnnData to {out_file}")


Saved AnnData to /group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis/bal_shared.h5ad
Saved AnnData to /group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis/pbmc_shared.h5ad


## Subset + rename ann_lv2

In [7]:
# --- PBMC subset ---
pbmc_keep = ["CD14.mono", "CD16.mono"]
adata_pbmc_sub = adata_pbmc[adata_pbmc.obs[CLUSTER_KEY].isin(pbmc_keep)].copy()

def rename_pbmc(x: str) -> str:
    if x == "CD14.mono":
        return "PBMC_CD14.mono"
    if x == "CD16.mono":
        return "PBMC_CD16.mono"
    return None

adata_pbmc_sub.obs[CLUSTER_KEY] = adata_pbmc_sub.obs[CLUSTER_KEY].astype(str).map(rename_pbmc)
adata_pbmc_sub = adata_pbmc_sub[adata_pbmc_sub.obs[CLUSTER_KEY].notna()].copy()

# --- BAL subset ---
bal_lv2 = adata_bal.obs[CLUSTER_KEY].astype(str)
bal_keep_mask = bal_lv2.str.startswith("AM") | bal_lv2.str.startswith("MDM") | bal_lv2.eq("CD14.mono")
adata_bal_sub = adata_bal[bal_keep_mask].copy()

def rename_bal(x: str) -> str:
    # CD14.mono
    if x == "CD14.mono":
        return "BAL_mono"
    # AM
    if x == "AM-CCL":
        return "BAL_AM-CCL"
    if x.startswith("AM"):
        return "BAL_AM"
    # MDM
    if x == "MDM-PLA2G7hi":
        return "BAL_MDM-PLA2G7hi"
    if x.startswith("MDM"):
        return "BAL_MDM"
    return None

adata_bal_sub.obs[CLUSTER_KEY] = adata_bal_sub.obs[CLUSTER_KEY].astype(str).map(rename_bal)
adata_bal_sub = adata_bal_sub[adata_bal_sub.obs[CLUSTER_KEY].notna()].copy()

print("PBMC subset counts:")
print(adata_pbmc_sub.obs[CLUSTER_KEY].value_counts())
print("BAL subset counts:")
print(adata_bal_sub.obs[CLUSTER_KEY].value_counts())


PBMC subset counts:
ann_lv2
PBMC_CD14.mono    1131
PBMC_CD16.mono     655
Name: count, dtype: int64
BAL subset counts:
ann_lv2
BAL_AM              69183
BAL_AM-CCL           4177
BAL_MDM              4022
BAL_mono             2609
BAL_MDM-PLA2G7hi     2273
Name: count, dtype: int64


## Merge PBMC + BAL subsets

We concatenate with an outer join on genes. Downstream HVG selection will pick a shared set.

## Visualization

### Dotplot

In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset PBMC to CD14.mono [fibrosis gene]
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_pbmc[
    adata_pbmc.obs[ct_key_lv2].astype(str).isin(["CD14.mono"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "NRG1",
    "SPP1","CHI3L1","CHIT1","MMP9","PLA2G7","LGMN","CD163","MERTK"
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    #cmap="Blues",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD14.mono fibro", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC-CD14.mono")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC-CD14.mono_fibrodotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset PBMC to CD16.mono [fibrosis gene]
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_pbmc[
    adata_pbmc.obs[ct_key_lv2].astype(str).isin(["CD16.mono"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "NRG1",
    "SPP1","CHI3L1","CHIT1","MMP9","PLA2G7","LGMN","CD163","MERTK"
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    #cmap="Blues",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD16.mono fibro", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC-CD16.mono")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC-CD16.mono_fibrodotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset BAL to cDC
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_bal[
    adata_bal.obs[ct_key_lv2].astype(str).isin(["cDC"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "CCL4", "CCL3", "CCL2", "CXCL10", "CCL3L1",
    "IFI6", "IFIT3",
    "CTSL", "CTSB", "LGMN",
    "HLA-DQA2",
    "XCR1", "CLEC9A", "IRF8",
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    cmap="Blues",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("BAL cDC", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/BAL_cDC")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"BAL-cDC_DEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset BAL to cDC
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_pbmc[
    adata_pbmc.obs[ct_key_lv2].astype(str).isin(["CD14.mono"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = ["CXCL8","EREG","SERPINB2","F13A1","CDC42BPA","NRG1"]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD14.mono", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC_CD14.mono")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC_CD14.mono_DEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset BAL to DC migratory
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_bal[
    adata_bal.obs[ct_key_lv2].astype(str).isin(["DC.migratory"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = ["ITGB8","CTSL","CTSD","CTSB","A2M","APOE","CD68","LGALS3","FCER1G","CD1C","CCL17","FSCN1"]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    cmap="Blues",
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("BAL DC.migratory", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/BAL_DC.migratory")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"BAL-DC.migratory_DEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
